In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langchian_core.messages import BaseMessage,Human_message
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random 


ModuleNotFoundError: No module named 'langchian_core'

In [ ]:
load_dotenv()

In [ ]:
llm=ChatOllama(model="llama3.1:8b")


In [ ]:
search_tool=DuckDuckGoSearchRun(region="us-en")

@tool
def calcultor(first_num:float,second_num:float,operation:str)->dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations:add, sub, mul, div.
    """
    try:
        if operation=="add":
            result=first_num+second_num
        elif operation=="sub": 
            result=first_num-second_num
        elif operation=="mul":
            result=first_num*second_num
        elif operation=="div":
            if second_num==0:
                return {"error":"Division by zero is not allowed."}
            result=first_num/second_num
        else:
            return {"error":"Unsupported operation. Please use add, sub, mul, or div."}
        return {"first_num":first_num,"second_num":second_num,"operation":operation,"result":result}
    except Exception as e:
        return {"error":str(e)}

@tool
def get_stock_price(symbol:str)->dict:
    """
    Fetch the current stock price for a given symbol(eg: AAPL,TSLA).
    using Alpha Vantage with API key in the URL.
    """
    url=f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=A38M2IF3HPGODENI"
    r=requests.get(url)
    return r.json()




In [ ]:
tools=[calcultor,get_stock_price,search_tool]

llm_with_tools=llm.bind_tools(tools)


In [ ]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]


In [ ]:
def chat_node(state:ChatState):
    """
    LLM node that may answer or request a tool call.
    """
    messages=state["messages"]
    response=llm_with_tools.invoke(messages)
    return{"messages":[response]}

tool_node=ToolNode(tools=tools)

In [ ]:
graph=StateGraph(ChatState)

graph.add_node("chat_node",chat_node)
graph.add_node("tool_node",tool_node)

graph.add_edge(START,"chat_node")
graph.add_condition_edge("chat_node",tools_condition)

graph.add_edge("tool_node","chat_node")
graph.add_edge("tool_node",END)

In [ ]:
out=chatbot.invoke({"messages":[Human_message(content="What is the current stock price of AAPL and what is 5 multiplied by 3?")]})
print(out["messages"][-1].content)